# 05 · Risk score vs ToS;DR grades

Phase 9. Put your collected ToS texts in `data/tos_docs/` and fill in `data/tosdr_grades.csv` (see `data/tos_docs/README.md`). The data folder is in the repo, so commit the files, or upload them to `My Drive/fineguard/data/` and run the copy cell.

Reports Spearman ρ with a bootstrap 95% CI for **equal weights** (v1) and **fitted non-negative weights** (v2, leave-one-out).

## Setup
Clones the repo, installs packages, and links `models/` and `mlruns/` to Google Drive so nothing is lost if Colab disconnects.

In [ ]:
# 1. Get the code (for a private repo, see the README: add GITHUB_TOKEN in Colab Secrets)
import os
if not os.path.exists("/content/fineguard"):
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        !git clone -q https://{token}@github.com/424103-achu/fineguard.git /content/fineguard
    except Exception:
        !git clone -q https://github.com/424103-achu/fineguard.git /content/fineguard
%cd /content/fineguard/ml
!git pull -q
!pip install -q -r requirements.txt

In [ ]:
# 2. Save outputs to Google Drive so they survive a disconnect
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/fineguard"
!mkdir -p {DRIVE}/models {DRIVE}/mlruns {DRIVE}/figures
!rm -rf models mlruns && ln -sfn {DRIVE}/models models && ln -sfn {DRIVE}/mlruns mlruns

# MLflow's SQLite file works best on local disk: copy it from Drive now, back at the end
import os, shutil
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:////content/mlflow.db"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
if os.path.exists(f"{DRIVE}/mlruns/mlflow.db"):
    shutil.copy(f"{DRIVE}/mlruns/mlflow.db", "/content/mlflow.db")
FIG = f"{DRIVE}/figures"

In [ ]:
# Optional: use documents stored in Drive instead of the repo
import os
if os.path.exists(f"{DRIVE}/data/tosdr_grades.csv"):
    !cp -r {DRIVE}/data/tos_docs/. data/tos_docs/ && cp {DRIVE}/data/tosdr_grades.csv data/
!wc -l data/tosdr_grades.csv

## Validate the score
`--write-weights` saves the fitted v2 weights next to the model (`score_weights.json`); the web service loads them automatically.

In [ ]:
!python -m src.validate_score --model models/legal-bert-unfair --write-weights

In [ ]:
# Copy the MLflow database back to Drive (run this after every training cell you care about)
shutil.copy("/content/mlflow.db", f"{DRIVE}/mlruns/mlflow.db")
!python -m src.results_table --out {DRIVE}/results.md

## Score vs ToS;DR grade

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
df = pd.read_csv("data/score_validation.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
order = sorted(df.tosdr_grade.unique())
sns.boxplot(data=df, x="tosdr_grade", y="score_equal", order=order, ax=axes[0])
sns.stripplot(data=df, x="tosdr_grade", y="score_equal", order=order, color="k", ax=axes[0])
axes[0].set_title("Equal-weight risk score by ToS;DR grade")
sns.boxplot(data=df, x="tosdr_grade", y="loo_pred", order=order, ax=axes[1])
sns.stripplot(data=df, x="tosdr_grade", y="loo_pred", order=order, color="k", ax=axes[1])
axes[1].set_title("Fitted weights: leave-one-out prediction")
plt.tight_layout(); plt.savefig(f"{FIG}/score_vs_tosdr.png", dpi=150); plt.show()
df.to_csv(f"{DRIVE}/score_validation.csv", index=False)
df